In [ ]:
from pathlib import Path
import json
import pandas as pd
import pickle

BASE_DIR = Path("../data/processed/merged_participants.csv")

with open('../config/config.json') as f:
    config = json.load(f)

In [ ]:
df = pd.read_csv(BASE_DIR)
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
feature_cols = [c for c in df.columns if c != 'participant_id']
X = df[feature_cols]

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
kmeans = KMeans(
    n_clusters=config['model_settings']['n_clusters'],
    random_state=config['model_settings']['random_state'],
    n_init=config['model_settings']['n_init']
)

df['cluster'] = kmeans.fit_predict(X_scaled)
silhouette = silhouette_score(X_scaled, df["cluster"])
print("Silhouetter score: ", round(silhouette, 3))

In [ ]:
cluster_means = df.groupby('cluster')[["fixation_mean", "n_saccades"]].mean()
print("\nCluster centers (fixation_mean & n_saccades): ")
print(cluster_means)

In [ ]:
holistic_cluster = cluster_means["fixation_mean"].idxmax() if cluster_means.loc[cluster_means["fixation_mean"].idxmax(), "n_saccades"] < cluster_means["n_saccades"].mean() else cluster_means["n_saccades"].idxmin()

df['strategy'] = df['cluster'].apply(lambda x: 'Holistic' if x == holistic_cluster else 'Piecemeal')

print("\nStrategy assigned:")
print(df[['participant_id', 'cluster', 'strategy']])

In [ ]:
with open('../src/models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

print('KMeans model saved successfully using pickle!')